In [178]:
import pandas as pd

In [179]:
df = pd.read_csv('jeju.csv',encoding='cp949')
df.head(2)

,지역,읍면동,전화번호,주소,일반현황,청사현황,자치센터현황,데이터기준일자,Unnamed: 8
0,제주시,일도1동,064-728-4412,제주특별자치도 제주시 중앙로7길 15 (일도일동),인구 3146명 / 면적 0.33㎢ / 11통 61개반,지상 3층 / 연면적 585㎡,개소일 2000-12-20,2019-06-20,NaN
1,제주시,일도2동,064-728-4442,제주특별자치도 제주시 고마로 28 (일도이동),인구 38400명 / 면적2.18㎢ / 48통326반,지상 2층·지하 1층 / 연면적 1307.01,총면적 617.22㎡ / 개소일 2000-12-24,2019-06-20,NaN


In [180]:
#1. 주소 정보를 이용해서 위도, 경도 값을 저장합니다. '위치'컬럼에 (위도,경도) 저장
# 주소를 찾아서 저장하는 함수를 작성하여 처리
# 함수 이름은 geo_coder(query) 리턴값 : (위도,경도) 만일 못찾았다면 '결과없음'으로 리턴
# 데이터프레임적용할때 .apply()사용

In [181]:
def geo_coder(query):
    import requests
    REST_API_KEY = 'ba78bc9aa277131240f5fa32dab22e71'
    query = query.split('(')[0]
    url = f"https://dapi.kakao.com/v2/local/search/keyword.json?category_group_code=PO3&query={query}"
    headers = {'Authorization':f'KakaoAK {REST_API_KEY}'}
    res = requests.get(url,headers=headers)
    jsondata = res.json()
    # print(query)
    # print(jsondata)
    try:
        return jsondata['documents'][0]['y'],jsondata['documents'][0]['x']
    except:
        return '결과없음'

In [182]:
df['위치'] = df['주소'].apply(geo_coder)

In [183]:
df['위치'].head()

0    (33.5150737361162, 126.526342790758)
1                                    결과없음
2    (33.5069402044009, 126.526976882749)
3    (33.4970469392912, 126.535290947586)
4    (33.5041154034239, 126.517362400655)
Name: 위치, dtype: object

In [184]:
df.head(5)

,지역,읍면동,전화번호,주소,일반현황,청사현황,자치센터현황,데이터기준일자,Unnamed: 8,위치
0,제주시,일도1동,064-728-4412,제주특별자치도 제주시 중앙로7길 15 (일도일동),인구 3146명 / 면적 0.33㎢ / 11통 61개반,지상 3층 / 연면적 585㎡,개소일 2000-12-20,2019-06-20,NaN,"(33.5150737361162, 126.526342790758)"
1,제주시,일도2동,064-728-4442,제주특별자치도 제주시 고마로 28 (일도이동),인구 38400명 / 면적2.18㎢ / 48통326반,지상 2층·지하 1층 / 연면적 1307.01,총면적 617.22㎡ / 개소일 2000-12-24,2019-06-20,NaN,결과없음
2,제주시,이도1동,064-728-1530,제주특별자치도 제주시 중앙로25길 17 (이도1동),인구 7418명 / 면적 0.79㎢ / 16통 97개반,지상 2층 / 2층면적 205㎡,개소일 2000-12-28,2019-06-20,NaN,"(33.5069402044009, 126.526976882749)"
3,제주시,이도2동,064-755-2021,제주특별자치도 제주시 오복3길 8 (이도이동),인구 41428명 / 면적 5.4㎢ / 49통 323반,지하 1층·지상 3층 / 연면적 1223.52㎡,총면적 388.9㎡ / 개소일 2000-10-05,2019-06-20,NaN,"(33.4970469392912, 126.535290947586)"
4,제주시,삼도1동,064-728-4532,제주특별자치도 제주시 전농로 40 (삼도일동),인구 14281명 / 면적0.87㎢ / 18통122반,지하 1층·지상3층 / 연면적 2070.37㎡,총면적 456.55㎡,2019-06-20,NaN,"(33.5041154034239, 126.517362400655)"


In [185]:
#2 '위치' 컬럼의 값을 Marker 위치값으로 , tooltip으로 읍면동을 사용하여 지도에 표시(클리스터이용)
# 함수사용, apply()를 이용해 데이터 프레임에 적용

In [186]:
import folium
from folium.plugins import MarkerCluster

In [187]:
def display(loc,map):
    print(loc[0],loc[1])
    if loc[0] != '결과없음':
        folium.Marker(loc[0],tooltip=loc[1]).add_to(map)
    

In [188]:
map = folium.Map((33.4996213, 126.5311884), zoom_start=8)
map_c = MarkerCluster().add_to(map)
df[['위치','읍면동']].apply(display,map=map_c,axis=1)
map

('33.5150737361162', '126.526342790758') 일도1동
결과없음 일도2동
('33.5069402044009', '126.526976882749') 이도1동
('33.4970469392912', '126.535290947586') 이도2동
('33.5041154034239', '126.517362400655') 삼도1동
('33.5117364099539', '126.522210779319') 삼도2동
('33.50909779859489', '126.51326647018216') 용담1동
('33.5114717601895', '126.511681792482') 용담2동
('33.5150321080548', '126.531518045311') 건입동
('33.52021737989457', '126.5654667308753') 화북동
('33.5218857941584', '126.585599365991') 삼양동
('33.4917232671377', '126.594687244151') 봉개동
('33.4763364526145', '126.545269320539') 아라동
('33.49512911933711', '126.5115798618361') 오라동
('33.4881587279011', '126.496886364509') 연동
('33.4830787815432', '126.477191625104') 노형동
('33.4928558848117', '126.432175190874') 외도동
('33.4997987984561', '126.458099340466') 이호동
('33.5029008833987', '126.468230562279') 도두동
('33.410671262551865', '126.26694451906893') 한림읍
결과없음 애월읍
('33.52252490132381', '126.85205144932898') 구좌읍
('33.53440130090102', '126.63413223748915') 조천읍
('33.30042738

In [189]:
map = folium.Map((33.4996213, 126.5311884), zoom_start=8)
map_c = MarkerCluster().add_to(map)
for index, row in df.iterrows():
    # print(index)
    # print(row['읍면동'],row['위치'])
    if row['위치'] != '결과없음':
        folium.Marker(row['위치'],tooltip=row['읍면동']).add_to(map_c)
map

In [190]:
#3. '일반현황' 컬럼의 인구수만 분리하여 '인구수'라는 int형 컬럼을 생성

In [191]:
df['인구수'] = df['일반현황'].str.extract('(\d+)').astype(int)
# '일반현황' 컬럼에서 숫자만 추출하여 '인구수'라는 새로운 컬럼을 생성
#  str.extract('(\d+)') '일반현황' 컬럼에서 숫자만 추출하는 정규표현식
df

,지역,읍면동,전화번호,주소,일반현황,청사현황,자치센터현황,데이터기준일자,Unnamed: 8,위치,인구수
0,제주시,일도1동,064-728-4412,제주특별자치도 제주시 중앙로7길 15 (일도일동),인구 3146명 / 면적 0.33㎢ / 11통 61개반,지상 3층 / 연면적 585㎡,개소일 2000-12-20,2019-06-20,NaN,"(33.5150737361162, 126.526342790758)",3146
1,제주시,일도2동,064-728-4442,제주특별자치도 제주시 고마로 28 (일도이동),인구 38400명 / 면적2.18㎢ / 48통326반,지상 2층·지하 1층 / 연면적 1307.01,총면적 617.22㎡ / 개소일 2000-12-24,2019-06-20,NaN,결과없음,38400
2,제주시,이도1동,064-728-1530,제주특별자치도 제주시 중앙로25길 17 (이도1동),인구 7418명 / 면적 0.79㎢ / 16통 97개반,지상 2층 / 2층면적 205㎡,개소일 2000-12-28,2019-06-20,NaN,"(33.5069402044009, 126.526976882749)",7418
3,제주시,이도2동,064-755-2021,제주특별자치도 제주시 오복3길 8 (이도이동),인구 41428명 / 면적 5.4㎢ / 49통 323반,지하 1층·지상 3층 / 연면적 1223.52㎡,총면적 388.9㎡ / 개소일 2000-10-05,2019-06-20,NaN,"(33.4970469392912, 126.535290947586)",41428
4,제주시,삼도1동,064-728-4532,제주특별자치도 제주시 전농로 40 (삼도일동),인구 14281명 / 면적0.87㎢ / 18통122반,지하 1층·지상3층 / 연면적 2070.37㎡,총면적 456.55㎡,2019-06-20,NaN,"(33.5041154034239, 126.517362400655)",14281
5,제주시,삼도2동,064-728-4562,제주특별자치도 제주시 관덕로6길 15 (삼도이동),인구 9431명 / 면적0.83㎢ / 19통109반,지하 1층·지상3층 / 연면적 814㎡,총면적 368.69㎡ / 개소일 2000-12-30,2019-06-20,NaN,"(33.5117364099539, 126.522210779319)",9431
6,제주시,용담1동,064-728-4592,제주특별자치도 제주시 용남3길 11 (용담일동),인구 8368명 / 면적0.61㎢ / 15통102반,지상 3층 / 대지면적 8825㎡ / 건축면적 428.55,연면적 112049㎡ / 개소일 2000-12-24,2019-06-20,NaN,"(33.50909779859489, 126.51326647018216)",8368
7,제주시,용담2동,064-728-1535,제주특별자치도 제주시 흥운길 27 (용담이동),인구 16701명 / 면적 4.94㎢ / 23통 155반,지상 3층 지하 1층 / 연면적 1046.13㎡,총면적 449.55㎡ / 개관일 2000-12-30,2019-06-20,NaN,"(33.5114717601895, 126.511681792482)",16701
8,제주시,건입동,064-728-1536,제주특별자치도 제주시 만덕로 18 (건입동),인구 10850명 / 면적2.5㎢ / 20통113반,지상 4층 / 연면적 1192.37㎡,총면적 323.92㎡ / 개소일 2000-12-27,2019-06-20,NaN,"(33.5150321080548, 126.531518045311)",10850
9,제주시,화북동,064-728-4682,제주특별자치도 제주시 진남로6길 17 (화북일동),인구 25559('17.5월 현재) / 면적8.28㎢ / 33통 205반,지하 1층·지상3층 / 연면적 2375.80㎡,연면적 54㎡ / 개소일 2000-12-27,2019-06-20,NaN,"(33.52021737989457, 126.5654667308753)",25559


In [192]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   지역          43 non-null     object 
 1   읍면동         43 non-null     object 
 2   전화번호        43 non-null     object 
 3   주소          43 non-null     object 
 4   일반현황        43 non-null     object 
 5   청사현황        43 non-null     object 
 6   자치센터현황      41 non-null     object 
 7   데이터기준일자     43 non-null     object 
 8   Unnamed: 8  0 non-null      float64
 9   위치          43 non-null     object 
 10  인구수         43 non-null     int32  
dtypes: float64(1), int32(1), object(9)
memory usage: 3.7+ KB


In [193]:
df['일반현황'].str.split('/').str.get(0).str.split('(').str.get(0).str.split().str.get(-1).str.replace('명','').astype(int)

0      3146
1     38400
2      7418
3     41428
4     14281
5      9431
6      8368
7     16701
8     10850
9     25559
10    25000
11     3046
12    28741
13    13729
14    39897
15    56223
16    17606
17     4104
18     2837
19    19925
20    26539
21    15080
22    20804
23     8745
24     2820
25     1718
26    16610
27    19151
28    13866
29    11867
30    11167
31     5207
32     2444
33     4003
34     3798
35     5475
36     4980
37    22225
38    10697
39    10344
40    11310
41    10684
42     3813
Name: 일반현황, dtype: int32

In [198]:
import re
# df['인구수'] = df['일반현황'].apply(lambda x:int(re.findall('[0-9]{4,}',x)[-1]))
df['일반현황'].apply(lambda x:int(re.findall('[0-9]{4,}',x)[-1]))

0      3146
1     38400
2      7418
3     41428
4     14281
5      9431
6      8368
7     16701
8     10850
9     25559
10    25000
11     3046
12    28741
13    13729
14    39897
15    56223
16    17606
17     4104
18     2837
19    19925
20    26539
21    15080
22    20804
23    79123
24     2820
25     1718
26    16610
27    19151
28    13866
29    11867
30    11167
31     5207
32     2444
33     4003
34     3798
35     5475
36     4980
37    22225
38    10697
39    10344
40    11310
41    10684
42     3813
Name: 일반현황, dtype: int64

In [199]:
df.head(2)

,지역,읍면동,전화번호,주소,일반현황,청사현황,자치센터현황,데이터기준일자,Unnamed: 8,위치,인구수
0,제주시,일도1동,064-728-4412,제주특별자치도 제주시 중앙로7길 15 (일도일동),인구 3146명 / 면적 0.33㎢ / 11통 61개반,지상 3층 / 연면적 585㎡,개소일 2000-12-20,2019-06-20,NaN,"(33.5150737361162, 126.526342790758)",3146
1,제주시,일도2동,064-728-4442,제주특별자치도 제주시 고마로 28 (일도이동),인구 38400명 / 면적2.18㎢ / 48통326반,지상 2층·지하 1층 / 연면적 1307.01,총면적 617.22㎡ / 개소일 2000-12-24,2019-06-20,NaN,결과없음,38400
